# tuning hyperparamter


3种调参方法：
1. 手动
2. grid search： 会建立一些超参数对网格，逐个尝试，效率低下
3. random search: 随机组合超参数网格值，
4. auto tuning: 贝叶斯优化等

调参往往是一个极其耗时的过程

我们会划分训练集和验证集，以评价参数

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import  train_test_split

N_FOLDS = 5
MAX_EVALS = 5


In [3]:
features = pd.read_csv('data/application_train.csv')
features = features.sample(n=16000)

In [4]:
features = features.select_dtypes('number')

In [14]:
labels = np.array(features['TARGET'].astype(np.int32))
features = features.drop(columns = ['SK_ID_CURR', 'TARGET'])

In [15]:
train_features, test_features, train_labels, test_labels = train_test_split(features, labels, test_size = 6000, random_state = 50)

## CV

In [16]:
train_set = lgb.Dataset(data=train_features, label=train_labels)
test_set = lgb.Dataset(data=test_features, label=test_labels)

In [17]:
model = lgb.LGBMClassifier()

In [19]:
default_params = model.get_params()
default_params

{'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 1.0,
 'importance_type': 'split',
 'learning_rate': 0.1,
 'max_depth': -1,
 'min_child_samples': 20,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 100,
 'n_jobs': None,
 'num_leaves': 31,
 'objective': None,
 'random_state': None,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0,
 'subsample': 1.0,
 'subsample_for_bin': 200000,
 'subsample_freq': 0}

In [20]:
help(lgb.cv)

Help on function cv in module lightgbm.engine:

cv(params: Dict[str, Any], train_set: lightgbm.basic.Dataset, num_boost_round: int = 100, folds: Union[Iterable[Tuple[numpy.ndarray, numpy.ndarray]], sklearn.model_selection._split.BaseCrossValidator, NoneType] = None, nfold: int = 5, stratified: bool = True, shuffle: bool = True, metrics: Union[str, List[str], NoneType] = None, feval: Union[Callable[[numpy.ndarray, lightgbm.basic.Dataset], Tuple[str, float, bool]], Callable[[numpy.ndarray, lightgbm.basic.Dataset], List[Tuple[str, float, bool]]], List[Union[Callable[[numpy.ndarray, lightgbm.basic.Dataset], Tuple[str, float, bool]], Callable[[numpy.ndarray, lightgbm.basic.Dataset], List[Tuple[str, float, bool]]]]], NoneType] = None, init_model: Union[str, pathlib.Path, lightgbm.basic.Booster, NoneType] = None, fpreproc: Optional[Callable[[lightgbm.basic.Dataset, lightgbm.basic.Dataset, Dict[str, Any]], Tuple[lightgbm.basic.Dataset, lightgbm.basic.Dataset, Dict[str, Any]]]] = None, seed: in

In [22]:
cv_results = lgb.cv(
    default_params,
    train_set,
    metrics = 'auc',
    num_boost_round=10000,
    nfold=N_FOLDS,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 如果连续200迭代没有提升auc，就自动停止 
    ]
)

[LightGBM] [Warning] Unknown parameter: importance_type
[LightGBM] [Warning] Unknown parameter: importance_type
[LightGBM] [Warning] Unknown parameter: importance_type
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007028 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9966
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 93
[LightGBM] [Warning] Unknown parameter: importance_type
[LightGBM] [Warning] Unknown parameter: importance_type
[LightGBM] [Warning] Unknown parameter: importance_type
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003572 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9966
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 93
[LightGBM] [Warning] Unknown parameter: importance_type
[LightGBM] [Warning] Unknown par

In [23]:
cv_results

{'valid auc-mean': [0.65323335141481,
  0.6775583880504371,
  0.6817425581169888,
  0.6873531227080611,
  0.6898062112949138,
  0.6939104626960922,
  0.6957445684624461,
  0.6966996317241883,
  0.699616249256568,
  0.6992604166394905,
  0.7009808396041705,
  0.7013666166946605,
  0.7032957304365313,
  0.7060287134596595],
 'valid auc-stdv': [0.024272781101888612,
  0.01812150742035603,
  0.024102039428268655,
  0.019996301858114098,
  0.021825004316099888,
  0.020519177494757675,
  0.025953258778170885,
  0.021477480378160406,
  0.02144838863473038,
  0.018759257870372713,
  0.01978071914621832,
  0.019988042613973512,
  0.0220677550289575,
  0.022302328855071645]}